#Sistema Inteligente de Consulta Bibliográfica

**Autor:** Jorge Fuentes
**Tecnologías principales:** Python, Pandas, SQLite3, LangChain, Google Gemini API y Streamlit.



### Objetivo del Proyecto
Los catálogos bibliográficos históricos suelen almacenarse en archivos planos masivos con formatos inconsistentes, valores nulos y ruido tipográfico, lo que dificulta su explotación por parte de usuarios no técnicos. 

El objetivo de este proyecto es construir un **pipeline de datos completo (End-to-End)** capaz de transformar un dataset crudo de más de **2,3 millones de registros** de la Biblioteca Nacional de España (BNE) en una base de datos optimizada, e integrarla con un **Agente de Inteligencia Artificial (Text-to-SQL)** que permita consultar el catálogo en lenguaje natural desde una interfaz web interactiva.



### Arquitectura y Flujo de Trabajo
El cuaderno se divide en 4 fases principales:

1. **Exploración y Limpieza de Datos:**
   * Análisis exploratorio sobre un dataset inicial de **2.368.294 registros** y **35 columnas** (~1.4 GB en memoria).
   * Eliminación de registros críticos incompletos (libros sin título) y descarte de 14 columnas con metadatos administrativos irrelevantes o exceso de nulos ('id_symphony', 'deposito_legal', 'nipo', etc.).
   * Normalización de fechas con formatos corruptos ('19uu', '19--', 'uuuu', '?') e imputación de valores centinela ('-99999' en campos numéricos y 'Desconocido' en campos de texto) para garantizar la integridad de tipos (int64 / str).


2. **Creación de la Base de Datos SQL (`biblioteca.db`):**
   * Migración del DataFrame limpio a un motor de base de datos relacional local (`biblioteca.db`) para permitir consultas rápidas mediante SQL.
  
     
3. **Motor de Inteligencia Artificial (*Text-to-SQL* con LangChain & Gemini):**
   * Conexión del esquema SQLite con el modelo **`gemini-3.8-flash`**.
   * Diseño de *Prompt Engineering* con reglas de negocio y control de calidad de datos (filtrado automático de valores centinela '-99999', exclusión de errores tipográficos de catalogación y omisión de etiquetas 'Desconocido' en agrupaciones).
   * Generación automática de consultas SQL, ejecución en tiempo real y síntesis de respuestas en lenguaje natural.

   
4. **Despliegue con Streamlit:**
   * Creación automatizada de la aplicación `app.py` para ofrecer un buscador interactivo de cara al usuario final.

---

**IMPORTANTE**: En la siguiente celda se debe poner la ruta de ordenador del archivo csv con los datos.

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns


ruta =r"C:\Users\Jorge\Desktop\mis experimentos\monomodernas-UTF8\mon\monomodernas-UTF8.csv" #Se debe modificar esta ruta dependiendo de donde tengas el archivo .csv
libros= pd.read_csv(ruta, sep=';', low_memory=False)

pd.set_option('display.max_columns', None)
libros.head()


,id,id_symphony,id_datos_bne,per_id,pais_de_publicacion,lengua_principal,otras_lenguas,lengua_original,fecha_de_publicacion,decada,siglo,deposito_legal,isbn,nipo,cdu,autores,titulo,mencion_de_autores,otros_titulos,edicion,lugar_de_publicacion,editorial,extension,otras_caracteristicas_fisicas,dimensiones,material_anejo,serie,nota_de_contenido,notas,procedencia,premios,tema,genero_forma,tipo_de_documento,url
0,991061093353808606,NaN,NaN,NaN,España,español,NaN,NaN,2024,20.0,XXI,B 14555-2024,978-84-19399-06-9,NaN,NaN,NaN,Franco:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registro provisional,NaN,NaN,NaN,NaN,Monografía en papel (posterior a 1830),NaN
1,991061093353708606,NaN,NaN,NaN,España,español,NaN,NaN,2024,20.0,XXI,CS 719-2024,NaN,NaN,NaN,NaN,Peña Ciclista Excursionista:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registro provisional,NaN,NaN,NaN,NaN,Monografía en papel (posterior a 1830),NaN
2,991061093353608606,NaN,NaN,NaN,España,español,NaN,NaN,2024,20.0,XXI,BI 559-2024,978-84-19670-88-5,NaN,NaN,NaN,Coronel Weird y el pequeño Andrómeda:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registro provisional,NaN,NaN,NaN,NaN,Monografía en papel (posterior a 1830),NaN
3,991061093353508606,NaN,NaN,NaN,España,español,NaN,NaN,2024,20.0,XXI,CS 48-2024,978-84-127230-6-9,NaN,NaN,NaN,La Vergine del Pilar tras San Giacomo e San Pa...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registro provisional,NaN,NaN,NaN,NaN,Monografía en papel (posterior a 1830),NaN
4,991061093353408606,NaN,NaN,NaN,España,español,NaN,NaN,2024,20.0,XXI,CS 750-2024,NaN,NaN,NaN,NaN,I Congrés d'història de Nules:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registro provisional,NaN,NaN,NaN,NaN,Monografía en papel (posterior a 1830),NaN


In [3]:
libros.info()

<class 'pandas.DataFrame'>
RangeIndex: 2368294 entries, 0 to 2368293
Data columns (total 35 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   id                             int64  
 1   id_symphony                    str    
 2   id_datos_bne                   object 
 3   per_id                         str    
 4   pais_de_publicacion            str    
 5   lengua_principal               str    
 6   otras_lenguas                  str    
 7   lengua_original                str    
 8   fecha_de_publicacion           str    
 9   decada                         float64
 10  siglo                          str    
 11  deposito_legal                 str    
 12  isbn                           str    
 13  nipo                           str    
 14  cdu                            str    
 15  autores                        str    
 16  titulo                         str    
 17  mencion_de_autores             str    
 18  otros_titulos

---
**1. EXPLORACIÓN Y LIMPIEZA DE DATOS**

In [4]:
conteo_nans = libros.isna().sum()
print(conteo_nans)

id                                     0
id_symphony                        77561
id_datos_bne                       45801
per_id                           2352709
pais_de_publicacion                 1352
lengua_principal                     329
otras_lenguas                    2356511
lengua_original                  2034431
fecha_de_publicacion               39115
decada                             40862
siglo                              39115
deposito_legal                    689662
isbn                             1013769
nipo                             2347530
cdu                               344885
autores                           679300
titulo                                77
mencion_de_autores                544173
otros_titulos                    2250550
edicion                          1709821
lugar_de_publicacion               96216
editorial                          56172
extension                          75470
otras_caracteristicas_fisicas    1404973
dimensiones     

---
Vamos a empezar con la limpieza. 
En primer lugar vemos como hay 77 libros sin titulo, los cuales no nos sirven para nada por lo que los podemos borrar

In [5]:
libros=libros.dropna(subset=["titulo"])

**preparacion de la fecha de publicacion**

veamos si podemos convertir la fecha_de_publicacion en enteros y si hay cosas raras saber cuales 

In [6]:
fechas_convertidas = pd.to_numeric(libros['fecha_de_publicacion'], errors='coerce')


valores_no_numericos = libros[libros['fecha_de_publicacion'].notna() & fechas_convertidas.isna()]
conteo_malos = valores_no_numericos['fecha_de_publicacion'].value_counts()

print(f"Total de valores con formato extraño: {len(valores_no_numericos)}")

display(conteo_malos.head(50))

Total de valores con formato extraño: 3811


fecha_de_publicacion
s202    606
uuuu    556
19uu    533
19--    430
195u    243
194u    133
193u    130
199-    127
192u    111
18uu     97
198u     66
197u     58
196u     56
191u     55
189u     47
198-     47
s201     45
?        39
199u     33
200-     32
200u     30
195-     28
201u     24
188u     24
190u     21
17uu     18
184u     17
20uu     17
185u     14
194-     14
193-     11
196-     11
19??     10
197-     10
192-      9
16uu      6
191-      6
|s20      6
189-      5
194?      5
201-      5
195?      4
d         4
n         4
197?      3
s         3
187u      3
a         3
184-      3
||||      2
Name: count, dtype: int64

---
Vemos que hay 3811 casos en los que no podemos convertirlo en enteros por lo que vamos a modificarlos para que puedan haciendo una aproximacion lo dependiendo de los casos. 

* Los casos que tenemos menos de 20 libros directamente los vamos a poner como NaN ya que no pasa nada por perder esa pequeña cantidad de datos

* En los casos de 'uuuu', '?', 's202', 's201','19uu','19--','18uu' como no podemos hacer una buena aproximación entonces vamos a ponerlos como NaN

* En los casos que tengan una sola 'u' o un '-' vamos a sustituirlos con un 0


In [7]:
raros_menos_de_20 = conteo_malos[conteo_malos < 20].index


libros['fecha_de_publicacion'] = libros['fecha_de_publicacion'].replace(raros_menos_de_20, np.nan)

In [8]:
desconocidos = ['uuuu', '?', 's202', 's201','19uu','19--','18uu']
libros['fecha_de_publicacion'] = libros['fecha_de_publicacion'].replace(desconocidos, np.nan)



libros['fecha_de_publicacion'] = (libros['fecha_de_publicacion'].astype(str).str.replace('u', '0', regex=False).str.replace('-', '0', regex=False))



comprobamos ejecutando el codigo de antes para ver si ya no tenemos ningun valor que no sea numerico

In [9]:
fechas_convertidas = pd.to_numeric(libros['fecha_de_publicacion'], errors='coerce')


valores_no_numericos = libros[libros['fecha_de_publicacion'].notna() & fechas_convertidas.isna()]
conteo_malos = valores_no_numericos['fecha_de_publicacion'].value_counts()

print(f"Total de valores con formato extraño: {len(valores_no_numericos)}")
display(conteo_malos.head(50))

Total de valores con formato extraño: 0


Series([], Name: count, dtype: int64)

Ahora en vamos a rellenar los nulos con la palabra "desconocico" o un valor especifico en las columnas de autores, lugar de publicacion, editorial, fecha de publicacion, decada, siglo, pais de publicacion, lengua principal, otras lenguas, ISBN, cdu, extension, dimensiones, tema y forma de genero.

In [10]:
libros['autores'] = libros['autores'].fillna('Desconocido')
libros['lugar_de_publicacion'] = libros['lugar_de_publicacion'].fillna('Desconocido')
libros['editorial'] = libros['editorial'].fillna('Desconocida')
libros['fecha_de_publicacion'] = libros['fecha_de_publicacion'].fillna(-99999).astype(int)
libros['decada'] = libros['decada'].fillna(-99999).astype(int) #le pongo -99999 porque al ser números no se confunda si ponemos 0 con la decada de los 2000
libros['siglo']=libros['siglo'].fillna('Desconocida')
libros['pais_de_publicacion']=libros['pais_de_publicacion'].fillna('Desconocido')
libros['lengua_principal']=libros['lengua_principal'].fillna('Desconocida')
libros['otras_lenguas']=libros['otras_lenguas'].fillna('No')
libros['otras_lenguas']=libros['otras_lenguas'].fillna('No')
libros['isbn']=libros['isbn'].fillna('Desconocido')
libros['cdu'] = libros['cdu'].fillna('Desconocido')
libros['extension'] = libros['extension'].fillna('Desconocida')
libros['dimensiones'] = libros['dimensiones'].fillna('Desconocido')
libros['tema'] = libros['tema'].fillna('Desconocido')
libros['genero_forma'] = libros['genero_forma'].fillna('Desconocido')

vemos cuales son las que nos quedan: 

In [11]:
conteo_nans = libros.isna().sum()
print(conteo_nans)

id                                     0
id_symphony                        77561
id_datos_bne                       45801
per_id                           2352632
pais_de_publicacion                    0
lengua_principal                       0
otras_lenguas                          0
lengua_original                  2034356
fecha_de_publicacion                   0
decada                                 0
siglo                                  0
deposito_legal                    689654
isbn                                   0
nipo                             2347453
cdu                                    0
autores                                0
titulo                                 0
mencion_de_autores                544096
otros_titulos                    2250473
edicion                          1709745
lugar_de_publicacion                   0
editorial                              0
extension                              0
otras_caracteristicas_fisicas    1404899
dimensiones     

Veamos que vamos a hacer con cada una de las columnas que nos quedan sin arreglar:
* id_symphony, id_datos_bne, id_datos_bne, per_id: lo vamos a eliminar ya que son codigos/identificadores interno de la biblioteca de donde se han sacado los datos y no nos interesa
* lengua_original: en este caso los valores nulos es porque la original y la principal son la misma por lo que los valores nulos seran una copia del valor que tengamos en la pricipal
* deposito_legal: lo vamos a eliminar ya que es una obligación establecida por ley en muchos países que obliga al editor a entregar uno o varios ejemplares de cualquier publicación a una institución pública.
* nipo: es el código obligatorio que se asigna en España a todas las publicaciones editadas por la Administración General del Estado pero como nuestro buscador no queremos esta informacion lo eliminamos
* mencion de autores: como son datos adicionales sobre dterminados libros los que no tengasn pondremos que no tienen
* otros titulos: en esta columna está el titulo en otros idiomas pero como no lo necesitamos la borramos
* edicion: nos da la informacion de cuando salio la primera edicion pero eso ya lo tenemos en la columna fecha por lo que podemos borrarla
* otras_caracteristicas_fisicas: son datos que no nos interesan
* material_anejo: se refiere a si venia con material añadido como un tríptico o un cd por lo que lo podemos borrar
* serie: se refiere a que tematica pertenece por lo que son datos interesantes y aunque haya muchos nulos nos interesan
* nota_de_contenido: suele tener textos muy largos con la forma que tiene el libro por dentro pero como quiero crear una ia de consulta no quiero que profuncice en la estructura interna del libro por lo que la voy a borrar
* notas: son datos sobre quien ha hecho la traduccion y ciertas caracteristicas del libro pero en la mayoria de casos son datos irrelevantes por lo que se puede borrar
* procendencia, premios y url: son datos nulos en casi todas las filas por lo que lo mejor es borrarlos

In [12]:
libros = libros.drop(columns=['id_symphony','id_datos_bne','per_id','deposito_legal','nipo','otros_titulos','edicion','otras_caracteristicas_fisicas','material_anejo','nota_de_contenido','notas','procedencia','premios','url'])

libros['lengua_original'] = libros['lengua_original'].fillna(libros['lengua_principal'])


libros['mencion_de_autores'] = libros['mencion_de_autores'].fillna('Sin mención de autores')

libros['serie'] = libros['serie'].fillna('Sin serie')

In [13]:
conteo_nans = libros.isna().sum()
print(conteo_nans)

id                      0
pais_de_publicacion     0
lengua_principal        0
otras_lenguas           0
lengua_original         0
fecha_de_publicacion    0
decada                  0
siglo                   0
isbn                    0
cdu                     0
autores                 0
titulo                  0
mencion_de_autores      0
lugar_de_publicacion    0
editorial               0
extension               0
dimensiones             0
serie                   0
tema                    0
genero_forma            0
tipo_de_documento       0
dtype: int64


In [14]:
libros.info()

<class 'pandas.DataFrame'>
Index: 2368217 entries, 0 to 2368293
Data columns (total 21 columns):
 #   Column                Dtype
---  ------                -----
 0   id                    int64
 1   pais_de_publicacion   str  
 2   lengua_principal      str  
 3   otras_lenguas         str  
 4   lengua_original       str  
 5   fecha_de_publicacion  int64
 6   decada                int64
 7   siglo                 str  
 8   isbn                  str  
 9   cdu                   str  
 10  autores               str  
 11  titulo                str  
 12  mencion_de_autores    str  
 13  lugar_de_publicacion  str  
 14  editorial             str  
 15  extension             str  
 16  dimensiones           str  
 17  serie                 str  
 18  tema                  str  
 19  genero_forma          str  
 20  tipo_de_documento     str  
dtypes: int64(3), str(18)
memory usage: 1.1 GB


---
**2. CREACIÓN DE LA BASE DE DATOS SQL**

In [15]:
import sqlite3

In [16]:
conexion = sqlite3.connect('biblioteca.db')
libros.to_sql('libros', conexion, if_exists='replace', index=False)
conexion.close()

print("Tabla libros creada con éxito en la base de datos")

Tabla libros creada con éxito en la base de datos


Comprobamos que se haya creado bien

In [18]:
conexion = sqlite3.connect('biblioteca.db')

consulta_prueba = pd.read_sql("SELECT titulo, fecha_de_publicacion, lengua_principal FROM libros LIMIT 5", conexion)
display(consulta_prueba)

conexion.close()

,titulo,fecha_de_publicacion,lengua_principal
0,Franco:,2024,español
1,Peña Ciclista Excursionista:,2024,español
2,Coronel Weird y el pequeño Andrómeda:,2024,español
3,La Vergine del Pilar tras San Giacomo e San Pa...,2024,español
4,I Congrés d'història de Nules:,2024,español


In [19]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///biblioteca.db")

print("Esquema detectado por LangChain:")
print(db.get_table_info())

C:\Users\Jorge\AppData\Local\Temp\ipykernel_22596\3544624554.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Esquema detectado por LangChain:

CREATE TABLE libros (
	id INTEGER, 
	pais_de_publicacion TEXT, 
	lengua_principal TEXT, 
	otras_lenguas TEXT, 
	lengua_original TEXT, 
	fecha_de_publicacion INTEGER, 
	decada INTEGER, 
	siglo TEXT, 
	isbn TEXT, 
	cdu TEXT, 
	autores TEXT, 
	titulo TEXT, 
	mencion_de_autores TEXT, 
	lugar_de_publicacion TEXT, 
	editorial TEXT, 
	extension TEXT, 
	dimensiones TEXT, 
	serie TEXT, 
	tema TEXT, 
	genero_forma TEXT, 
	tipo_de_documento TEXT
)

/*
3 rows from libros table:
id	pais_de_publicacion	lengua_principal	otras_lenguas	lengua_original	fecha_de_publicacion	decada	siglo	isbn	cdu	autores	titulo	mencion_de_autores	lugar_de_publicacion	editorial	extension	dimensiones	serie	tema	genero_forma	tipo_de_documento
991061093353808606	España	español	No	español	2024	20	XXI	978-84-19399-06-9	Desconocido	Desconocido	Franco:	Sin mención de autores	Desconocido	Desconocida	Desconocida	Desconocido	Sin serie	Desconocido	Desconocido	Monografía en papel (posterior a 1830)
99

---
**3. MOTOR DE INTELIGENCIA ARTIFICIAL**


PARA PROBAR SI FUNCIONA LA CONSULTA CON EL AGENTE DE IA SE DEBE MARCAR LA **SIGUIENTE CELDA COMO CODIGO** Y AÑADIR LA API Key DE GEMINI DONDE PONE "PONER API KEY PARA PROBAR"

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser

os.environ["GOOGLE_API_KEY"] = "PONER API KEY PARA PROBAR "

db = SQLDatabase.from_uri("sqlite:///biblioteca.db")

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
cadena_texto = llm | StrOutputParser()

def consultar_biblioteca(pregunta):
    schema = db.get_table_info()
    
    #Genera la consulta SQL
    prompt_sql = f"""
                Basándote en el siguiente esquema de base de datos SQLite:
                {schema}

                REGLAS CRÍTICAS DE LA BASE DE DATOS:
                1. En 'fecha_de_publicacion', el valor -99999 es nulo y los números menores a 1500 (como 9, 20, 130) son errores tipográficos de catalogación. SIEMPRE que busques libros antiguos, ordenes fechas o hagas cálculos temporales, filtra únicamente años históricos reales usando: WHERE fecha_de_publicacion BETWEEN 1500 AND 2026.
                2. En columnas de texto, 'Desconocido', 'Desconocida', 'Sin serie' y 'Sin mención de autores' representan valores nulos. Exclúyelos siempre que te pidan rankings, conteos o listas (ej. WHERE autores != 'Desconocido').

                Escribe únicamente una consulta SQL válida de SQLite que responda a esta pregunta: "{pregunta}"
                No incluyas explicaciones ni bloques markdown, solo la sentencia SELECT limpia.
                """
    
    sql_query = cadena_texto.invoke(prompt_sql)
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    print(f"SQL generado: {sql_query}")
    
    # Ejecuta la consulta en SQLite
    resultado_db = db.run(sql_query)
    print(f"Datos obtenidos: {resultado_db}")
    
    #Redacta la respuesta final en lenguaje natural
    prompt_respuesta = f"""
    Pregunta del usuario: "{pregunta}"
    Consulta SQL ejecutada: {sql_query}
    Resultado de la base de datos: {resultado_db}
    
    Responde a la pregunta del usuario en español de forma clara, natural y directa basándote en el resultado.
    """
    
    respuesta_final = cadena_texto.invoke(prompt_respuesta)
    print(f"\nRespuesta:\n{respuesta_final}")
    return respuesta_final


consultar_biblioteca("¿Cuántos libros hay en total que se hayan publicado desde el año 2000 en la base de datos?")

---
**4. DESPLIEGUE CON STREAMLIT**

En el streamlit vamos a poner que en la parte izquierda esté una pestaña de escritura para que la persona que quiera utilizar el asistente escriba su API Key de Gemini

Además, pondremos en esta seción 3 ejemplos de preguntas que se le puede hacer al agente IA

En cuanto al agente buscador le pondremos una condiciones para que no muestre ejemplos erroneos:

* Si se pregunta sobre la fecha de publicacion entonces hay que tomar el valor -9999 como nulo y los nuemeros menores de 1500 como errores tipógráficos en la catalogación.
* En las columnas de texto 'Desconocido', 'Desconocida', 'Sin serie' y 'Sin mención de autores' las tome como representacion de valores nulos
* Finalmente que al mostras la consukta SQL no incluya explicaciones, solo la sentencia SELECT limpia.


In [20]:
import streamlit as st

In [21]:
%%writefile app.py
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser

st.set_page_config(page_title="Asistente de Biblioteca IA", layout="centered")
st.title("Asistente Inteligente de Biblioteca")
st.markdown("Haz preguntas en lenguaje natural sobre el catálogo de más de **2,3 millones de libros**.")

with st.sidebar:
    st.header("Configuración")
    api_key = st.text_input("Introduce tu Google API Key:", type="password")
    st.markdown("---")
    st.markdown("**Ejemplos de preguntas:**")
    st.caption("• ¿Cuáles son los 5 libros más antiguos y de qué año son?")
    st.caption("• ¿Cuáles son los 5 autores con más libros?")
    st.caption("• ¿Cuántos libros hay publicados desde el año 2000?")

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key
    db = SQLDatabase.from_uri("sqlite:///biblioteca.db")
    llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
    cadena_texto = llm | StrOutputParser()

    pregunta = st.text_input("Escribe tu pregunta:", placeholder="Ej: ¿Cuáles son los libros más antiguos?")

    if st.button("Consultar catálogo") and pregunta:
        with st.spinner("Generando consulta SQL y buscando en la base de datos..."):
            try:
                schema = db.get_table_info()
                
                prompt_sql = f"""
                Basándote en el siguiente esquema de base de datos SQLite:
                {schema}

                REGLAS CRÍTICAS DE LA BASE DE DATOS:
                1. En 'fecha_de_publicacion', el valor -99999 es nulo y los números menores a 1500 (como 9, 20, 130) son errores tipográficos de catalogación. SIEMPRE que busques libros antiguos, ordenes fechas o hagas cálculos temporales, filtra únicamente años históricos reales usando: WHERE fecha_de_publicacion BETWEEN 1500 AND 2026.
                2. En columnas de texto, 'Desconocido', 'Desconocida', 'Sin serie' y 'Sin mención de autores' representan valores nulos. Exclúyelos siempre que te pidan rankings, conteos o listas (ej. WHERE autores != 'Desconocido').

                Escribe únicamente una consulta SQL válida de SQLite que responda a esta pregunta: "{pregunta}"
                No incluyas explicaciones ni bloques markdown, solo la sentencia SELECT limpia.
                """
                sql_query = cadena_texto.invoke(prompt_sql).replace("```sql", "").replace("```", "").strip()
                
                with st.expander("Ver consulta SQL generada"):
                    st.code(sql_query, language="sql")
                
                resultado_db = db.run(sql_query)
                
                with st.expander("Ver datos crudos de SQLite"):
                    st.write(resultado_db)
                
                prompt_respuesta = f"""
                Pregunta del usuario: "{pregunta}"
                Consulta SQL ejecutada: {sql_query}
                Resultado de la base de datos: {resultado_db}
                
                Responde a la pregunta del usuario en español de forma clara, natural y directa basándote en el resultado.
                """
                respuesta_final = cadena_texto.invoke(prompt_respuesta)
                
                st.success("¡Respuesta encontrada!")
                st.markdown(f"### Resultado:\n{respuesta_final}")
                
            except Exception as e:
                st.error(f"Ocurrió un error al procesar la consulta: {e}")
else:
    st.info("Por favor, pega tu Google API Key en el menú de la izquierda para activar el asistente.")

Overwriting app.py
